# Optimization System - pontos de mínimo e máximo

Problema: encontrar os pontos de máximo e mínimo de

$$f(x,y)=(x-1)^2+y^2-2$$

sujeita à restrição

$$x^2+y^2=1.$$

O notebook contém:
- solução analítica usada apenas para validação;
- visualização da função e da região viável;
- Algoritmo Genético com codificação real e reparo por projeção;
- PSO com a mesma estratégia de reparo;
- ajuste simples de parâmetros;
- comparação entre GA e PSO com o mesmo critério de parada.


In [6]:
import math
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "font.size": 11,
})


## 1. Solução analítica

Como a restrição é $x^2+y^2=1$, a função objetivo na circunferência pode ser simplificada:

$$f(x,y)=x^2-2x+1+y^2-2=(x^2+y^2)-2x-1=-2x.$$

Logo, minimizar $f$ equivale a maximizar $x$, e maximizar $f$ equivale a minimizar $x$ na circunferência unitária.

- Mínimo: $(x,y)=(1,0)$, com $f=-2$.
- Máximo: $(x,y)=(-1,0)$, com $f=2$.

Esses valores não são fornecidos aos algoritmos. Eles aparecem apenas na validação dos resultados.


In [7]:
def f_xy(x, y):
    return (x - 1) ** 2 + y**2 - 2


def constraint_residual_xy(x, y):
    return x**2 + y**2 - 1


def constraint_violation_xy(x, y):
    return np.abs(constraint_residual_xy(x, y))


analytic = pd.DataFrame([
    {"tipo": "mínimo", "x": 1.0, "y": 0.0, "f(x,y)": f_xy(1.0, 0.0)},
    {"tipo": "máximo", "x": -1.0, "y": 0.0, "f(x,y)": f_xy(-1.0, 0.0)},
])
analytic


,tipo,x,y,"f(x,y)"
0,mínimo,1.0,0.0,-2.0
1,máximo,-1.0,0.0,2.0


## 2. Gráfico 3D

A superfície colorida representa a função objetivo. A circunferência preta mostra os pontos viáveis $x^2+y^2=1$ sobre a superfície $z=f(x,y)$.


In [8]:
grid = np.linspace(-1.6, 1.6, 140)
X, Y = np.meshgrid(grid, grid)
Z = f_xy(X, Y)

theta = np.linspace(0, 2 * np.pi, 300)
circle_x = np.cos(theta)
circle_y = np.sin(theta)
circle_z = f_xy(circle_x, circle_y)

theta_cyl = np.linspace(0, 2 * np.pi, 120)
z_cyl = np.linspace(Z.min(), Z.max(), 80)
Theta_cyl, Z_cyl = np.meshgrid(theta_cyl, z_cyl)
X_cyl = np.cos(Theta_cyl)
Y_cyl = np.sin(Theta_cyl)

fig = go.Figure()
fig.add_trace(go.Surface(
    x=X_cyl,
    y=Y_cyl,
    z=Z_cyl,
    opacity=0.18,
    colorscale="Blues",
    showscale=False,
    name="Restrição x²+y²=1",
))
fig.add_trace(go.Surface(
    x=X,
    y=Y,
    z=Z,
    colorscale="Viridis",
    opacity=0.82,
    colorbar=dict(title="f(x,y)"),
    name="f(x,y)",
))
fig.add_trace(go.Scatter3d(
    x=circle_x,
    y=circle_y,
    z=circle_z,
    mode="lines",
    line=dict(color="black", width=6),
    name="Restrição sobre f",
))
fig.add_trace(go.Scatter3d(
    x=[1],
    y=[0],
    z=[-2],
    mode="markers+text",
    marker=dict(size=6, color="green"),
    text=["mínimo"],
    textposition="middle right",
    name="Mínimo",
))
fig.add_trace(go.Scatter3d(
    x=[-1],
    y=[0],
    z=[2],
    mode="markers+text",
    marker=dict(size=6, color="red"),
    text=["máximo"],
    textposition="middle left",
    name="Máximo",
))
fig.update_layout(
    title="Função objetivo e curva viável",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="f(x,y)",
        aspectmode="cube",
    ),
    width=900,
    height=700,
)
fig.show()


In [9]:
grid = np.linspace(-1.6, 1.6, 160)
X, Y = np.meshgrid(grid, grid)
Z = f_xy(X, Y)

eps = 0.10
theta = np.linspace(0, 2 * np.pi, 400)
r = np.linspace(1 - eps, 1 + eps, 50)
Theta_band, R_band = np.meshgrid(theta, r)
X_band = R_band * np.cos(Theta_band)
Y_band = R_band * np.sin(Theta_band)
Z_band = f_xy(X_band, Y_band)

x_curve = np.cos(theta)
y_curve = np.sin(theta)
z_curve = f_xy(x_curve, y_curve)

x_min, y_min = 1.0, 0.0
x_max, y_max = -1.0, 0.0
z_min = f_xy(x_min, y_min)
z_max = f_xy(x_max, y_max)
z_base = np.min(Z) - 0.15

fig = go.Figure()
fig.add_trace(go.Surface(
    x=X,
    y=Y,
    z=Z,
    colorscale="Viridis",
    opacity=0.22,
    showscale=False,
    name="Superfície completa",
))
fig.add_trace(go.Surface(
    x=X_band,
    y=Y_band,
    z=Z_band,
    colorscale="Plasma",
    opacity=0.95,
    colorbar=dict(title="f(x,y)"),
    name="Fatia ao redor da restrição",
))
fig.add_trace(go.Scatter3d(
    x=x_curve,
    y=y_curve,
    z=z_curve,
    mode="lines",
    line=dict(color="black", width=8),
    name="Curva viável",
))
fig.add_trace(go.Scatter3d(
    x=x_curve,
    y=y_curve,
    z=np.full_like(theta, z_base),
    mode="lines",
    line=dict(color="gray", width=4, dash="dash"),
    name="Projeção da restrição",
))
fig.add_trace(go.Scatter3d(
    x=[x_min, x_min, None, x_max, x_max],
    y=[y_min, y_min, None, y_max, y_max],
    z=[z_base, z_min, None, z_base, z_max],
    mode="lines",
    line=dict(color="gray", width=4, dash="dot"),
    showlegend=False,
))
fig.add_trace(go.Scatter3d(
    x=[x_min],
    y=[y_min],
    z=[z_min],
    mode="markers+text",
    marker=dict(size=8, color="green"),
    text=["mínimo global"],
    textposition="top center",
    name="Mínimo",
))
fig.add_trace(go.Scatter3d(
    x=[x_max],
    y=[y_max],
    z=[z_max],
    mode="markers+text",
    marker=dict(size=8, color="red"),
    text=["máximo global"],
    textposition="top center",
    name="Máximo",
))
fig.update_layout(
    title="Fatia da superfície ao redor da restrição x² + y² = 1",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="f(x,y)",
        aspectmode="cube",
        camera=dict(eye=dict(x=1.6, y=1.4, z=1.1)),
    ),
    width=980,
    height=760,
    template="plotly_white",
)
fig.show()


## 3. Formulação computacional: reparo por projeção

Os algoritmos mantêm uma representação cartesiana real. Um indivíduo ou partícula bruta é $C=(x,y)$ e pode violar a restrição. Antes de cada avaliação, aplica-se o operador de reparo

$$C_{proj}=\frac{C}{\lVert C\rVert}, \qquad \lVert C\rVert=\sqrt{x^2+y^2}.$$

Para $C=(0,0)$, uma direção viável aleatória é gerada para evitar divisão por zero. A função objetivo e a fitness são calculadas somente em $C_{proj}$.

Esta projeção é um **operador de reparo de restrição**, não uma parametrização angular. O cromossomo continua sendo $(x,y)$ e os operadores trabalham no espaço cartesiano. O fluxo geral é:

$$\text{solução bruta}\rightarrow\text{reparo}\rightarrow\text{solução viável}\rightarrow\text{avaliação}.$$

Essa organização pode ser reutilizada com outros operadores de reparo e outras regiões viáveis, enquanto uma parametrização por $\theta$ seria específica para a circunferência.


In [10]:
BOUNDS = np.array([[-1.5, 1.5], [-1.5, 1.5]], dtype=float)
ZERO_NORM_EPS = 1e-14

COMMON_MAX_STEPS = 80
COMMON_PATIENCE = 15
COMMON_TOL = 1e-10


def repair_projection(points, rng=None):
    """Projeta pontos cartesianos na circunferência unitária.

    Este é um operador de reparo: os pontos brutos continuam sendo a
    representação usada pelos algoritmos. Não há parametrização por ângulo.
    """
    array = np.asarray(points, dtype=float)
    if array.shape[-1] != 2:
        raise ValueError("Cada ponto deve possuir exatamente duas coordenadas.")

    repaired = array.reshape(-1, 2).copy()
    norms = np.linalg.norm(repaired, axis=1)
    zero_mask = norms <= ZERO_NORM_EPS

    if np.any(zero_mask):
        generator = rng if rng is not None else np.random.default_rng(SEED)
        angles = generator.uniform(0.0, 2.0 * np.pi, size=np.count_nonzero(zero_mask))
        repaired[zero_mask, 0] = np.cos(angles)
        repaired[zero_mask, 1] = np.sin(angles)
        norms[zero_mask] = 1.0

    repaired /= norms[:, None]
    return repaired.reshape(array.shape)


def raw_objective(points):
    points = np.asarray(points, dtype=float)
    return f_xy(points[..., 0], points[..., 1])


def evaluate_with_projection(points, mode="min", rng=None):
    """Repara os pontos e retorna fitness, pontos viáveis e objetivo.

    A implementação maximiza fitness: para máximo, fitness=f; para mínimo,
    fitness=-f. Portanto, seleção e atualização sempre procuram maior fitness.
    """
    if mode not in {"min", "max"}:
        raise ValueError("mode deve ser 'min' ou 'max'.")

    projected = repair_projection(points, rng=rng)
    objective_values = raw_objective(projected)
    fitness = objective_values if mode == "max" else -objective_values
    return fitness, projected, objective_values


def summarize_solution(
    raw_point,
    projected_point,
    mode,
    evaluations,
    elapsed,
    history,
    history_evaluations,
):
    raw_point = np.asarray(raw_point, dtype=float)
    projected_point = np.asarray(projected_point, dtype=float)
    objective_value = float(raw_objective(projected_point))

    return {
        "modo": mode,
        "x_bruto": float(raw_point[0]),
        "y_bruto": float(raw_point[1]),
        "x_projetado": float(projected_point[0]),
        "y_projetado": float(projected_point[1]),
        "x": float(projected_point[0]),
        "y": float(projected_point[1]),
        "f(x,y)": objective_value,
        "fitness": objective_value if mode == "max" else -objective_value,
        "violacao_antes": float(constraint_violation_xy(raw_point[0], raw_point[1])),
        "violacao_depois": float(
            constraint_violation_xy(projected_point[0], projected_point[1])
        ),
        "avaliacoes": int(evaluations),
        "tempo_s": float(elapsed),
        "historico": np.asarray(history, dtype=float),
        "historico_avaliacoes": np.asarray(history_evaluations, dtype=int),
    }


## 4. Algoritmo Genético

Representação: cromossomo real com dois genes $(x,y)$.

Operadores:
- seleção por torneio;
- cruzamento aritmético;
- mutação gaussiana por gene;
- elitismo configurável;
- clipping apenas nos limites da caixa de busca;
- reparo por projeção antes de cada avaliação;
- parada antecipada por `patience`.

Os indivíduos não precisam nascer viáveis. Cruzamento e mutação produzem pontos cartesianos brutos; a projeção fornece a versão viável usada para objetivo e fitness.


In [11]:
@dataclass
class GAParams:
    population_size: int = 50
    generations: int = COMMON_MAX_STEPS
    crossover_rate: float = 0.80
    mutation_rate: float = 0.20
    mutation_sigma: float = 0.15
    elitism: int = 2
    tournament_size: int = 3
    patience: int = COMMON_PATIENCE
    tol: float = COMMON_TOL


def run_ga(mode="min", params=GAParams(), seed=SEED):
    local_rng = np.random.default_rng(seed)
    low, high = BOUNDS[:, 0], BOUNDS[:, 1]
    population = local_rng.uniform(low, high, size=(params.population_size, 2))
    initial_population_raw = population.copy()
    initial_population_projected = repair_projection(initial_population_raw, rng=local_rng)

    evaluations = 0
    history = []
    history_evaluations = []
    best_fitness_seen = -np.inf
    stale_generations = 0
    generations_used = 0
    start = time.perf_counter()

    def tournament(fitness):
        candidates = local_rng.integers(
            0,
            params.population_size,
            size=params.tournament_size,
        )
        winner = candidates[np.argmax(fitness[candidates])]
        return population[winner].copy()

    for generation in range(params.generations):
        fitness, projected_population, objective_values = evaluate_with_projection(
            population,
            mode=mode,
            rng=local_rng,
        )
        evaluations += len(population)
        generations_used = generation + 1

        best_index = int(np.argmax(fitness))
        current_best_fitness = float(fitness[best_index])
        history.append(float(objective_values[best_index]))
        history_evaluations.append(evaluations)

        if current_best_fitness - best_fitness_seen > params.tol:
            best_fitness_seen = current_best_fitness
            stale_generations = 0
        else:
            stale_generations += 1

        should_stop = (
            generation == params.generations - 1
            or (params.patience > 0 and stale_generations >= params.patience)
        )
        if should_stop:
            break

        elite_count = min(max(params.elitism, 0), params.population_size)
        elite_indices = np.argsort(fitness)[-elite_count:][::-1]
        next_population = [population[index].copy() for index in elite_indices]

        while len(next_population) < params.population_size:
            parent_1 = tournament(fitness)
            parent_2 = tournament(fitness)

            if local_rng.random() < params.crossover_rate:
                alpha = local_rng.random()
                child = alpha * parent_1 + (1.0 - alpha) * parent_2
            else:
                child = parent_1.copy()

            mutation_mask = local_rng.random(2) < params.mutation_rate
            mutation = local_rng.normal(0.0, params.mutation_sigma, size=2)
            child += mutation_mask * mutation
            child = np.clip(child, low, high)
            next_population.append(child)

        population = np.asarray(next_population, dtype=float)

    best_index = int(np.argmax(fitness))
    best_raw = population[best_index].copy()
    best_projected = projected_population[best_index].copy()
    elapsed = time.perf_counter() - start

    result = summarize_solution(
        best_raw,
        best_projected,
        mode,
        evaluations,
        elapsed,
        history,
        history_evaluations,
    )
    result["populacao_inicial_bruta"] = initial_population_raw
    result["populacao_inicial_projetada"] = initial_population_projected
    result["populacao_final_bruta"] = population.copy()
    result["populacao_final_projetada"] = projected_population.copy()
    result["geracoes_usadas"] = generations_used
    return result


## 5. PSO

Cada partícula também mantém uma posição cartesiana bruta $(x,y)$. A atualização segue:

$$v_i \leftarrow wv_i+c_1r_1(p_i-x_i)+c_2r_2(g-x_i)$$

$$x_i \leftarrow x_i+v_i.$$

Os limites da caixa de busca são mantidos por clipping. Antes de avaliar cada posição, aplica-se o mesmo reparo por projeção usado no GA. Os melhores pessoal e global são escolhidos pela fitness dos pontos projetados, embora a dinâmica continue armazenando posições brutas.


In [12]:
@dataclass
class PSOParams:
    swarm_size: int = 50
    iterations: int = COMMON_MAX_STEPS
    inertia: float = 0.72
    cognitive: float = 1.45
    social: float = 1.45
    vmax: float = 0.25
    patience: int = COMMON_PATIENCE
    tol: float = COMMON_TOL


def run_pso(mode="min", params=PSOParams(), seed=SEED):
    local_rng = np.random.default_rng(seed)
    low, high = BOUNDS[:, 0], BOUNDS[:, 1]
    position = local_rng.uniform(low, high, size=(params.swarm_size, 2))
    velocity = local_rng.uniform(
        -params.vmax,
        params.vmax,
        size=(params.swarm_size, 2),
    )

    initial_swarm_raw = position.copy()
    initial_swarm_projected = repair_projection(initial_swarm_raw, rng=local_rng)
    evaluations = 0
    history = []
    history_evaluations = []
    stale_iterations = 0
    iterations_used = 0
    start = time.perf_counter()

    fitness, projected_position, objective_values = evaluate_with_projection(
        position,
        mode=mode,
        rng=local_rng,
    )
    evaluations += params.swarm_size

    personal_best_raw = position.copy()
    personal_best_projected = projected_position.copy()
    personal_best_fitness = fitness.copy()
    personal_best_objective = objective_values.copy()

    global_index = int(np.argmax(personal_best_fitness))
    global_best_raw = personal_best_raw[global_index].copy()
    global_best_projected = personal_best_projected[global_index].copy()
    global_best_fitness = float(personal_best_fitness[global_index])
    global_best_objective = float(personal_best_objective[global_index])
    history.append(global_best_objective)
    history_evaluations.append(evaluations)

    for iteration in range(params.iterations):
        r1 = local_rng.random((params.swarm_size, 2))
        r2 = local_rng.random((params.swarm_size, 2))
        velocity = (
            params.inertia * velocity
            + params.cognitive * r1 * (personal_best_raw - position)
            + params.social * r2 * (global_best_raw - position)
        )
        velocity = np.clip(velocity, -params.vmax, params.vmax)
        position = np.clip(position + velocity, low, high)

        fitness, projected_position, objective_values = evaluate_with_projection(
            position,
            mode=mode,
            rng=local_rng,
        )
        evaluations += params.swarm_size
        iterations_used = iteration + 1

        improved = fitness > personal_best_fitness
        personal_best_raw[improved] = position[improved]
        personal_best_projected[improved] = projected_position[improved]
        personal_best_fitness[improved] = fitness[improved]
        personal_best_objective[improved] = objective_values[improved]

        candidate_index = int(np.argmax(personal_best_fitness))
        candidate_fitness = float(personal_best_fitness[candidate_index])
        improvement = candidate_fitness - global_best_fitness

        if candidate_fitness > global_best_fitness:
            global_best_raw = personal_best_raw[candidate_index].copy()
            global_best_projected = personal_best_projected[candidate_index].copy()
            global_best_fitness = candidate_fitness
            global_best_objective = float(personal_best_objective[candidate_index])

        if improvement > params.tol:
            stale_iterations = 0
        else:
            stale_iterations += 1

        history.append(global_best_objective)
        history_evaluations.append(evaluations)

        if params.patience > 0 and stale_iterations >= params.patience:
            break

    elapsed = time.perf_counter() - start
    result = summarize_solution(
        global_best_raw,
        global_best_projected,
        mode,
        evaluations,
        elapsed,
        history,
        history_evaluations,
    )
    result["enxame_inicial_bruto"] = initial_swarm_raw
    result["enxame_inicial_projetado"] = initial_swarm_projected
    result["enxame_final_bruto"] = position.copy()
    result["enxame_final_projetado"] = projected_position.copy()
    result["iteracoes_usadas"] = iterations_used
    return result


## 6. Ajuste dos parâmetros

A resposta analítica não participa da escolha dos parâmetros. Cada configuração é classificada pela própria fitness média obtida, com pesos pequenos para número de avaliações e tempo.

Para tornar a comparação justificável, GA e PSO usam o mesmo limite de 80 passos, a mesma paciência de 15 passos e a mesma tolerância de $10^{-10}$. A comparação gráfica também usa número de avaliações no eixo horizontal.


In [13]:
def score_solution(solution):
    return (
        -solution["fitness"]
        + 1e-7 * solution["avaliacoes"]
        + 1e-2 * solution["tempo_s"]
    )


ga_grid = [
    GAParams(
        population_size=population_size,
        generations=COMMON_MAX_STEPS,
        crossover_rate=crossover_rate,
        mutation_rate=0.20,
        mutation_sigma=mutation_sigma,
        elitism=2,
        tournament_size=3,
        patience=COMMON_PATIENCE,
        tol=COMMON_TOL,
    )
    for population_size in [30, 50]
    for crossover_rate in [0.80, 0.90]
    for mutation_sigma in [0.10, 0.20]
]

pso_grid = [
    PSOParams(
        swarm_size=swarm_size,
        iterations=COMMON_MAX_STEPS,
        inertia=0.72,
        cognitive=coefficient,
        social=coefficient,
        vmax=vmax,
        patience=COMMON_PATIENCE,
        tol=COMMON_TOL,
    )
    for swarm_size in [30, 50]
    for coefficient in [1.45, 1.65]
    for vmax in [0.20, 0.30]
]


def tune_algorithm(name, runner, grid, mode):
    rows = []
    for index, params in enumerate(grid):
        seeds = [SEED + 10 * index + repetition for repetition in range(2)]
        results = [runner(mode=mode, params=params, seed=seed) for seed in seeds]
        rows.append({
            "algoritmo": name,
            "modo": mode,
            "params": params,
            **vars(params),
            "objetivo_medio": np.mean([result["f(x,y)"] for result in results]),
            "fitness_media": np.mean([result["fitness"] for result in results]),
            "violacao_media_depois": np.mean(
                [result["violacao_depois"] for result in results]
            ),
            "tempo_medio_s": np.mean([result["tempo_s"] for result in results]),
            "avaliacoes": np.mean([result["avaliacoes"] for result in results]),
            "geracoes_usadas_media": np.mean(
                [result.get("geracoes_usadas", np.nan) for result in results]
            ),
            "iteracoes_usadas_media": np.mean(
                [result.get("iteracoes_usadas", np.nan) for result in results]
            ),
            "score": np.mean([score_solution(result) for result in results]),
        })
    return pd.DataFrame(rows).sort_values("score").reset_index(drop=True)


ga_tuning_min = tune_algorithm("GA", run_ga, ga_grid, "min")
ga_tuning_max = tune_algorithm("GA", run_ga, ga_grid, "max")
pso_tuning_min = tune_algorithm("PSO", run_pso, pso_grid, "min")
pso_tuning_max = tune_algorithm("PSO", run_pso, pso_grid, "max")

best_params = pd.concat([
    ga_tuning_min.head(1),
    ga_tuning_max.head(1),
    pso_tuning_min.head(1),
    pso_tuning_max.head(1),
], ignore_index=True)

metric_cols = [
    "objetivo_medio",
    "fitness_media",
    "violacao_media_depois",
    "tempo_medio_s",
    "avaliacoes",
    "geracoes_usadas_media",
    "iteracoes_usadas_media",
    "score",
]
fixed_cols = ["algoritmo", "modo", "params", *metric_cols]
param_cols = [column for column in best_params.columns if column not in fixed_cols]
best_params_view = best_params[
    ["algoritmo", "modo", *param_cols, *metric_cols]
].copy()

best_params_view


,algoritmo,modo,population_size,generations,crossover_rate,mutation_rate,mutation_sigma,elitism,tournament_size,patience,...,social,vmax,objetivo_medio,fitness_media,violacao_media_depois,tempo_medio_s,avaliacoes,geracoes_usadas_media,iteracoes_usadas_media,score
0,GA,min,30.0,80.0,0.8,0.2,0.2,2.0,3.0,15,...,NaN,NaN,-2.0,2.0,5.551115e-17,0.011119,795.0,26.5,NaN,-1.999809
1,GA,max,30.0,80.0,0.8,0.2,0.1,2.0,3.0,15,...,NaN,NaN,2.0,2.0,2.220446e-16,0.010088,720.0,24.0,NaN,-1.999827
2,PSO,min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15,...,1.65,0.2,-2.0,2.0,1.110223e-16,0.001061,1095.0,NaN,35.5,-1.999880
3,PSO,max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15,...,1.65,0.2,2.0,2.0,5.551115e-17,0.000643,750.0,NaN,24.0,-1.999919


In [14]:
# Seleção dos melhores parâmetros encontrados no ajuste

best_ga_min = ga_tuning_min.loc[0, "params"]
best_ga_max = ga_tuning_max.loc[0, "params"]
best_pso_min = pso_tuning_min.loc[0, "params"]
best_pso_max = pso_tuning_max.loc[0, "params"]

params_escolhidos = pd.DataFrame([
    {
        "algoritmo": "GA",
        "modo": mode,
        **vars(params),
    }
    for mode, params in [("min", best_ga_min), ("max", best_ga_max)]
] + [
    {
        "algoritmo": "PSO",
        "modo": mode,
        **vars(params),
    }
    for mode, params in [("min", best_pso_min), ("max", best_pso_max)]
])

params_escolhidos


,algoritmo,modo,population_size,generations,crossover_rate,mutation_rate,mutation_sigma,elitism,tournament_size,patience,tol,swarm_size,iterations,inertia,cognitive,social,vmax
0,GA,min,30.0,80.0,0.8,0.2,0.2,2.0,3.0,15,1.000000e-10,NaN,NaN,NaN,NaN,NaN,NaN
1,GA,max,30.0,80.0,0.8,0.2,0.1,2.0,3.0,15,1.000000e-10,NaN,NaN,NaN,NaN,NaN,NaN
2,PSO,min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15,1.000000e-10,30.0,80.0,0.72,1.65,1.65,0.2
3,PSO,max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15,1.000000e-10,30.0,80.0,0.72,1.65,1.65,0.2


## 7. Soluções obtidas com os melhores parâmetros

Os reinícios são comparados pela fitness produzida pelo próprio algoritmo. A solução analítica entra somente nas colunas de erro, depois que a otimização termina.


In [15]:
best_ga_min = ga_tuning_min.loc[0, "params"]
best_ga_max = ga_tuning_max.loc[0, "params"]
best_pso_min = pso_tuning_min.loc[0, "params"]
best_pso_max = pso_tuning_max.loc[0, "params"]


def best_restart(runner, mode, params, first_seed, repeats=5):
    candidates = [
        runner(mode=mode, params=params, seed=first_seed + index)
        for index in range(repeats)
    ]
    return max(candidates, key=lambda solution: solution["fitness"])


solutions = [
    {"algoritmo": "GA", **best_restart(run_ga, "min", best_ga_min, 101)},
    {"algoritmo": "GA", **best_restart(run_ga, "max", best_ga_max, 151)},
    {"algoritmo": "PSO", **best_restart(run_pso, "min", best_pso_min, 201)},
    {"algoritmo": "PSO", **best_restart(run_pso, "max", best_pso_max, 251)},
]

summary_cols = [
    "algoritmo",
    "modo",
    "x_bruto",
    "y_bruto",
    "x_projetado",
    "y_projetado",
    "f(x,y)",
    "f_exato",
    "erro_absoluto_f",
    "erro_ponto",
    "violacao_antes",
    "violacao_depois",
    "geracoes_usadas",
    "iteracoes_usadas",
    "avaliacoes",
    "tempo_s",
]

hidden_solution_fields = {
    "historico",
    "historico_avaliacoes",
    "populacao_inicial_bruta",
    "populacao_inicial_projetada",
    "populacao_final_bruta",
    "populacao_final_projetada",
    "enxame_inicial_bruto",
    "enxame_inicial_projetado",
    "enxame_final_bruto",
    "enxame_final_projetado",
}
summary = pd.DataFrame([
    {key: value for key, value in solution.items() if key not in hidden_solution_fields}
    for solution in solutions
])

# Valores analíticos usados somente para validar as soluções já encontradas.
summary["f_exato"] = summary["modo"].map({"min": -2.0, "max": 2.0})
summary["x_exato"] = summary["modo"].map({"min": 1.0, "max": -1.0})
summary["y_exato"] = 0.0
summary["erro_absoluto_f"] = np.abs(summary["f(x,y)"] - summary["f_exato"])
summary["erro_ponto"] = np.sqrt(
    (summary["x_projetado"] - summary["x_exato"]) ** 2
    + (summary["y_projetado"] - summary["y_exato"]) ** 2
)

summary_view = summary[summary_cols].copy()
coordinate_columns = ["x_bruto", "y_bruto", "x_projetado", "y_projetado"]
summary_view[coordinate_columns] = summary_view[coordinate_columns].round(8)
summary_view["f(x,y)"] = summary_view["f(x,y)"].round(10)
summary_view["f_exato"] = summary_view["f_exato"].round(10)

for column in [
    "erro_absoluto_f",
    "erro_ponto",
    "violacao_antes",
    "violacao_depois",
]:
    summary_view[column] = summary_view[column].map(lambda value: f"{value:.2e}")

for column in ["geracoes_usadas", "iteracoes_usadas"]:
    summary_view[column] = summary_view[column].map(
        lambda value: "-" if pd.isna(value) else int(value)
    )

summary_view["tempo_s"] = summary_view["tempo_s"].round(6)
summary_view


,algoritmo,modo,x_bruto,y_bruto,x_projetado,y_projetado,"f(x,y)",f_exato,erro_absoluto_f,erro_ponto,violacao_antes,violacao_depois,geracoes_usadas,iteracoes_usadas,avaliacoes,tempo_s
0,GA,min,0.781625,0.000000e+00,1.0,1.000000e-08,-2.0,-2.0,0.00e+00,6.27e-09,3.89e-01,0.00e+00,27,-,810,0.013889
1,GA,max,-0.369190,-0.000000e+00,-1.0,-0.000000e+00,2.0,2.0,0.00e+00,2.81e-09,8.64e-01,0.00e+00,28,-,840,0.010899
2,PSO,min,0.645496,9.000000e-08,1.0,1.400000e-07,-2.0,-2.0,1.91e-14,1.38e-07,5.83e-01,2.22e-16,-,41,1260,0.001007
3,PSO,max,-1.460046,6.800000e-07,-1.0,4.700000e-07,2.0,2.0,2.17e-13,4.66e-07,1.13e+00,0.00e+00,-,47,1440,0.001131


## 8. População inicial e final do GA

Cada painel mostra simultaneamente os cromossomos brutos e os pontos projetados usados na avaliação. Os segmentos cinza representam o deslocamento produzido pelo operador de reparo. Assim, fica explícito que o GA gera soluções no espaço cartesiano e só então as corrige para a circunferência.


In [16]:
def projection_segments(raw_points, projected_points):
    x_values = []
    y_values = []
    for raw, projected in zip(raw_points, projected_points):
        x_values.extend([raw[0], projected[0], None])
        y_values.extend([raw[1], projected[1], None])
    return x_values, y_values


def plot_ga_populations(solutions):
    ga_solutions = {
        solution["modo"]: solution
        for solution in solutions
        if solution["algoritmo"] == "GA"
    }
    theta = np.linspace(0, 2 * np.pi, 500)
    circle_x = np.cos(theta)
    circle_y = np.sin(theta)
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "GA mínimo - população inicial",
            "GA máximo - população inicial",
            "GA mínimo - população final",
            "GA máximo - população final",
        ),
    )

    for col, mode in enumerate(["min", "max"], start=1):
        solution = ga_solutions[mode]
        analytic_point = np.array([1.0, 0.0]) if mode == "min" else np.array([-1.0, 0.0])

        for row, stage in enumerate(["inicial", "final"], start=1):
            raw_points = np.asarray(solution[f"populacao_{stage}_bruta"])
            projected_points = np.asarray(solution[f"populacao_{stage}_projetada"])
            segment_x, segment_y = projection_segments(raw_points, projected_points)
            show_legend = row == 1 and col == 1

            fig.add_trace(go.Scatter(
                x=circle_x,
                y=circle_y,
                mode="lines",
                name="restrição",
                line=dict(color="black", width=2),
                legendgroup="restricao",
                showlegend=show_legend,
            ), row=row, col=col)
            fig.add_trace(go.Scatter(
                x=segment_x,
                y=segment_y,
                mode="lines",
                name="reparo",
                line=dict(color="rgba(100,100,100,0.25)", width=1),
                legendgroup="reparo",
                showlegend=show_legend,
            ), row=row, col=col)
            fig.add_trace(go.Scatter(
                x=raw_points[:, 0],
                y=raw_points[:, 1],
                mode="markers",
                name="pontos brutos",
                marker=dict(
                    color="#0077b6",
                    size=7,
                    opacity=0.55,
                    symbol="circle-open",
                ),
                legendgroup="brutos",
                showlegend=show_legend,
            ), row=row, col=col)
            fig.add_trace(go.Scatter(
                x=projected_points[:, 0],
                y=projected_points[:, 1],
                mode="markers",
                name="pontos projetados",
                marker=dict(color="#fb8500", size=7, opacity=0.75),
                legendgroup="projetados",
                showlegend=show_legend,
            ), row=row, col=col)

            if stage == "final":
                fig.add_trace(go.Scatter(
                    x=[solution["x_bruto"]],
                    y=[solution["y_bruto"]],
                    mode="markers",
                    name="melhor bruto",
                    marker=dict(color="#6a040f", size=11, symbol="diamond-open"),
                    legendgroup="melhor-bruto",
                    showlegend=(row == 2 and col == 1),
                ), row=row, col=col)
                fig.add_trace(go.Scatter(
                    x=[solution["x_projetado"]],
                    y=[solution["y_projetado"]],
                    mode="markers",
                    name="melhor projetado",
                    marker=dict(color="#d00000", size=13, symbol="star"),
                    legendgroup="melhor-projetado",
                    showlegend=(row == 2 and col == 1),
                ), row=row, col=col)
                fig.add_trace(go.Scatter(
                    x=[analytic_point[0]],
                    y=[analytic_point[1]],
                    mode="markers",
                    name="solução analítica",
                    marker=dict(color="#2d6a4f", size=11, symbol="x"),
                    legendgroup="analitica",
                    showlegend=(row == 2 and col == 1),
                ), row=row, col=col)

    fig.update_xaxes(title_text="x", range=[-1.6, 1.6], zeroline=True)
    fig.update_yaxes(title_text="y", range=[-1.6, 1.6], zeroline=True)
    for row in [1, 2]:
        fig.update_yaxes(scaleanchor=f"x{'' if row == 1 else 3}", scaleratio=1, row=row, col=1)
        fig.update_yaxes(scaleanchor=f"x{2 if row == 1 else 4}", scaleratio=1, row=row, col=2)
    fig.update_layout(
        title="GA: indivíduos brutos e reparados por projeção",
        width=1050,
        height=900,
        template="plotly_white",
        legend_title_text="Representação",
    )
    return fig


fig_ga_pop = plot_ga_populations(solutions)
fig_ga_pop.show()


## 9. Enxame inicial e final do PSO

Os painéis usam a mesma leitura do GA: posições brutas, projeções viáveis e segmentos de reparo. A dinâmica do PSO permanece no espaço cartesiano; apenas a avaliação usa os pontos projetados.


In [17]:
def plot_pso_swarms(solutions):
    pso_solutions = {
        solution["modo"]: solution
        for solution in solutions
        if solution["algoritmo"] == "PSO"
    }
    theta = np.linspace(0, 2 * np.pi, 500)
    circle_x = np.cos(theta)
    circle_y = np.sin(theta)
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "PSO mínimo - enxame inicial",
            "PSO máximo - enxame inicial",
            "PSO mínimo - enxame final",
            "PSO máximo - enxame final",
        ),
    )

    for col, mode in enumerate(["min", "max"], start=1):
        solution = pso_solutions[mode]
        analytic_point = np.array([1.0, 0.0]) if mode == "min" else np.array([-1.0, 0.0])

        for row, stage in enumerate(["inicial", "final"], start=1):
            raw_points = np.asarray(solution[f"enxame_{stage}_bruto"])
            projected_points = np.asarray(solution[f"enxame_{stage}_projetado"])
            segment_x, segment_y = projection_segments(raw_points, projected_points)
            show_legend = row == 1 and col == 1

            fig.add_trace(go.Scatter(
                x=circle_x,
                y=circle_y,
                mode="lines",
                name="restrição",
                line=dict(color="black", width=2),
                legendgroup="restricao",
                showlegend=show_legend,
            ), row=row, col=col)
            fig.add_trace(go.Scatter(
                x=segment_x,
                y=segment_y,
                mode="lines",
                name="reparo",
                line=dict(color="rgba(100,100,100,0.25)", width=1),
                legendgroup="reparo",
                showlegend=show_legend,
            ), row=row, col=col)
            fig.add_trace(go.Scatter(
                x=raw_points[:, 0],
                y=raw_points[:, 1],
                mode="markers",
                name="posições brutas",
                marker=dict(
                    color="#43aa8b",
                    size=7,
                    opacity=0.55,
                    symbol="circle-open",
                ),
                legendgroup="brutos",
                showlegend=show_legend,
            ), row=row, col=col)
            fig.add_trace(go.Scatter(
                x=projected_points[:, 0],
                y=projected_points[:, 1],
                mode="markers",
                name="posições projetadas",
                marker=dict(color="#577590", size=7, opacity=0.75),
                legendgroup="projetados",
                showlegend=show_legend,
            ), row=row, col=col)

            if stage == "final":
                fig.add_trace(go.Scatter(
                    x=[solution["x_bruto"]],
                    y=[solution["y_bruto"]],
                    mode="markers",
                    name="melhor bruto",
                    marker=dict(color="#6a040f", size=11, symbol="diamond-open"),
                    legendgroup="melhor-bruto",
                    showlegend=(row == 2 and col == 1),
                ), row=row, col=col)
                fig.add_trace(go.Scatter(
                    x=[solution["x_projetado"]],
                    y=[solution["y_projetado"]],
                    mode="markers",
                    name="melhor projetado",
                    marker=dict(color="#d00000", size=13, symbol="star"),
                    legendgroup="melhor-projetado",
                    showlegend=(row == 2 and col == 1),
                ), row=row, col=col)
                fig.add_trace(go.Scatter(
                    x=[analytic_point[0]],
                    y=[analytic_point[1]],
                    mode="markers",
                    name="solução analítica",
                    marker=dict(color="#2d6a4f", size=11, symbol="x"),
                    legendgroup="analitica",
                    showlegend=(row == 2 and col == 1),
                ), row=row, col=col)

    fig.update_xaxes(title_text="x", range=[-1.6, 1.6], zeroline=True)
    fig.update_yaxes(title_text="y", range=[-1.6, 1.6], zeroline=True)
    for row in [1, 2]:
        fig.update_yaxes(scaleanchor=f"x{'' if row == 1 else 3}", scaleratio=1, row=row, col=1)
        fig.update_yaxes(scaleanchor=f"x{2 if row == 1 else 4}", scaleratio=1, row=row, col=2)
    fig.update_layout(
        title="PSO: posições brutas e reparadas por projeção",
        width=1050,
        height=900,
        template="plotly_white",
        legend_title_text="Representação",
    )
    return fig


fig_pso_swarm = plot_pso_swarms(solutions)
fig_pso_swarm.show()


## 10. Convergência

O primeiro gráfico separa as quatro execuções. O segundo compara GA e PSO pelo número acumulado de avaliações, evitando que tamanhos diferentes de população ou enxame distorçam o eixo horizontal.


In [18]:
fig_individual = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "GA - mínimo",
        "GA - máximo",
        "PSO - mínimo",
        "PSO - máximo",
    ),
)

panel = {
    ("GA", "min"): (1, 1),
    ("GA", "max"): (1, 2),
    ("PSO", "min"): (2, 1),
    ("PSO", "max"): (2, 2),
}
colors = {"GA": "#0077b6", "PSO": "#e76f51"}

for solution in solutions:
    row, col = panel[(solution["algoritmo"], solution["modo"])]
    fig_individual.add_trace(go.Scatter(
        x=np.arange(len(solution["historico"])),
        y=solution["historico"],
        mode="lines+markers",
        line=dict(color=colors[solution["algoritmo"]]),
        marker=dict(size=4),
        name=f"{solution['algoritmo']} - {solution['modo']}",
        showlegend=False,
    ), row=row, col=col)

fig_individual.update_xaxes(title_text="geração/iteração")
fig_individual.update_yaxes(title_text="melhor f projetada")
fig_individual.update_layout(
    title="Convergência individual após reparo por projeção",
    width=1000,
    height=700,
    template="plotly_white",
)
fig_individual.show()

fig_comparison = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Comparação para mínimo", "Comparação para máximo"),
)

for col, mode in enumerate(["min", "max"], start=1):
    for solution in solutions:
        if solution["modo"] != mode:
            continue
        fig_comparison.add_trace(go.Scatter(
            x=solution["historico_avaliacoes"],
            y=solution["historico"],
            mode="lines+markers",
            name=solution["algoritmo"],
            legendgroup=solution["algoritmo"],
            showlegend=(col == 1),
            line=dict(color=colors[solution["algoritmo"]]),
            marker=dict(size=4),
        ), row=1, col=col)

fig_comparison.update_xaxes(title_text="avaliações acumuladas")
fig_comparison.update_yaxes(title_text="melhor f projetada")
fig_comparison.update_layout(
    title=(
        "GA vs PSO: mesmo limite de passos, patience e tolerância; "
        "comparação por avaliações"
    ),
    width=1000,
    height=430,
    template="plotly_white",
    legend_title_text="Algoritmo",
)
fig_comparison.show()


## 11. Pontos encontrados no plano $xy$

O gráfico reúne as quatro soluções projetadas e mostra o segmento entre cada melhor ponto bruto e sua versão reparada. Os marcadores analíticos servem somente como referência visual.


In [19]:
theta = np.linspace(0, 2 * np.pi, 600)
fig_points = go.Figure()
fig_points.add_trace(go.Scatter(
    x=np.cos(theta),
    y=np.sin(theta),
    mode="lines",
    name="x²+y²=1",
    line=dict(color="black", width=2),
))

solution_colors = {"GA": "#0077b6", "PSO": "#e76f51"}
solution_symbols = {"min": "triangle-down", "max": "triangle-up"}

for solution in solutions:
    fig_points.add_trace(go.Scatter(
        x=[solution["x_bruto"], solution["x_projetado"]],
        y=[solution["y_bruto"], solution["y_projetado"]],
        mode="lines",
        line=dict(color=solution_colors[solution["algoritmo"]], width=1, dash="dot"),
        showlegend=False,
    ))
    fig_points.add_trace(go.Scatter(
        x=[solution["x_projetado"]],
        y=[solution["y_projetado"]],
        mode="markers",
        name=f"{solution['algoritmo']} - {solution['modo']}",
        marker=dict(
            color=solution_colors[solution["algoritmo"]],
            size=13,
            symbol=solution_symbols[solution["modo"]],
        ),
    ))

fig_points.add_trace(go.Scatter(
    x=[1.0, -1.0],
    y=[0.0, 0.0],
    mode="markers+text",
    name="soluções analíticas",
    marker=dict(color="#2d6a4f", size=12, symbol="x"),
    text=["mínimo", "máximo"],
    textposition=["top right", "top left"],
))
fig_points.update_xaxes(title_text="x", range=[-1.6, 1.6], zeroline=True)
fig_points.update_yaxes(
    title_text="y",
    range=[-1.6, 1.6],
    zeroline=True,
    scaleanchor="x",
    scaleratio=1,
)
fig_points.update_layout(
    title="Soluções encontradas e circunferência viável",
    width=800,
    height=650,
    template="plotly_white",
)
fig_points.show()


## 12. Comparação e conclusões

GA e PSO trabalham com pontos cartesianos brutos que não precisam satisfazer $x^2+y^2=1$. Antes de cada avaliação, o operador `repair_projection` calcula

$$\lVert C\rVert=\sqrt{x^2+y^2}, \qquad C_{proj}=\frac{C}{\lVert C\rVert}.$$

Consequentemente,

$$x_{proj}^2+y_{proj}^2=1,$$

salvo o erro numérico de ponto flutuante. Quando o vetor bruto é nulo, o código gera uma direção unitária aleatória e evita a divisão por zero.

O principal diferencial do GA nesta implementação é o tratamento explícito da restrição:

- indivíduos reais podem ser gerados livremente no espaço cartesiano;
- cruzamento e mutação não precisam preservar viabilidade;
- cada descendente é reparado antes do cálculo de objetivo e fitness;
- a avaliação não depende de um coeficiente de penalidade;
- o padrão `gerar -> reparar -> avaliar` pode ser adaptado a outras regiões viáveis.

Os valores analíticos, mínimo $(1,0)$ com $f=-2$ e máximo $(-1,0)$ com $f=2$, foram usados apenas para verificar a qualidade das soluções após a execução.
